# 01 - Data Audit

## Customer Intelligence Platform

This notebook performs a comprehensive audit of the IBM Telco Customer Churn dataset.
We examine the dataset structure, identify data quality issues, assess potential leakage,
and document feature governance decisions.

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.load_data import load_raw
from src.config import LEAKAGE_COLUMNS, GEOGRAPHIC_COLUMNS, TARGET
from src.utils import df_summary


## 1. Dataset Overview

We begin by loading the raw dataset and understanding its structure.


In [ ]:
df = load_raw()
df_summary(df, "IBM Telco Customer Churn Dataset")


In [ ]:
print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Types:")
print(df.dtypes.value_counts())
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")


In [ ]:
df.head()


## 2. Column Classification

Every column must be classified by its role in the analysis:
- **Target**: The variable we're predicting
- **Feature**: Variables used for prediction
- **Identifier**: Unique keys (excluded from modeling)
- **Leakage**: Variables that encode the target or contain future information
- **Geographic**: Spatial data (loaded separately when needed)


In [ ]:
classification = []
for col in df.columns:
    if col == TARGET:
        role = "Target"
    elif col in LEAKAGE_COLUMNS:
        role = "Leakage / Drop"
    elif col in GEOGRAPHIC_COLUMNS:
        role = "Geographic"
    else:
        role = "Feature"

    classification.append({
        "Column": col,
        "Type": str(df[col].dtype),
        "Unique": df[col].nunique(),
        "Missing": df[col].isnull().sum(),
        "Role": role,
    })

class_df = pd.DataFrame(classification)
class_df


## 3. Missing Value Assessment


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing Count", ascending=False)
missing_df[missing_df["Missing Count"] > 0]


### Findings

1. **Churn Reason** has 5,174 missing values (73.5%). This is expected because churn reasons
   are only recorded for customers who have actually churned. This column is classified as leakage.
2. **No other columns** have missing values in their native state.
3. **Total Charges** appears complete but contains blank strings for 11 customers with 0 tenure.
   These will be converted to NaN during cleaning and imputed as $0.


## 4. Leakage Assessment

**Critical**: Identifying and removing leakage columns is essential for model validity.
These columns either directly encode the target or contain information unavailable at prediction time.


In [ ]:
print("Leakage Columns to Drop:")
print("-" * 40)
leakage_reasons = {
    "CustomerID": "Unique identifier, no predictive value",
    "Count": "Constant value (1 for every row)",
    "Country": "Constant value (United States)",
    "Lat Long": "Duplicate of Latitude/Longitude",
    "Churn Label": "Direct text encoding of target",
    "Churn Score": "Pre-computed churn score (leakage)",
    "CLTV": "IBM-computed CLV (contains future information)",
    "Churn Reason": "Only available after churn occurs",
}

for col, reason in leakage_reasons.items():
    print(f"  {col}: {reason}")


## 5. Target Variable Analysis


In [ ]:
print(f"Target Variable: {TARGET}")
print(f"\nValue Counts:")
print(df[TARGET].value_counts())
print(f"\nChurn Rate: {df[TARGET].mean():.2%}")
print(f"Class Ratio: {df[TARGET].value_counts()[0] / df[TARGET].value_counts()[1]:.2f}:1")


### Interpretation

The dataset contains **7,043 customers** with a churn rate of **26.54%** (approximately 1 in 4 customers).
This represents moderate class imbalance (2.77:1 ratio), which is manageable with standard techniques
like class weighting or threshold optimization.

---

## Summary

| Item | Value |
|------|-------|
| Total Records | 7,043 |
| Total Features | 33 |
| Target | Churn Value (binary 0/1) |
| Churn Rate | 26.54% |
| Leakage Columns | 8 (will be dropped) |
| Geographic Columns | 5 (loaded separately) |
| Missing Values | Churn Reason (73.5%), Total Charges (11 blank strings) |
